# Flipkart Customer Satisfaction Analysis and Prediction

**Author:** Aayush Kulkarni  
**Internship Organization:** Labmentix  
**Role:** Data Science Intern  
**Submission Date:** June 10, 2026  
**GitHub Repository:** https://github.com/kulkarniaayush90-sys/Flipkart-Customer-Satisfaction-Analysis-and-Prediction  
**Project Type:** Classification, Feature Engineering, Model Selection, Target Formulation Evaluation, and Deployment Planning


## Project Summary

This machine learning notebook completes the predictive modeling component of the **Flipkart Customer Satisfaction Analysis and Prediction** internship capstone project. The project uses approximately 85,907 customer support interactions to understand satisfaction outcomes and build models that can identify dissatisfied customers early enough for operational intervention. The notebook is standalone and submission-ready: it loads the dataset, audits data quality, engineers ML-ready features, handles missing and invalid values, encodes categorical variables, evaluates multiple classifiers, tunes the best model, analyzes feature importance, compares target formulations, and recommends a production deployment strategy.

The technical workflow converts raw support records into a structured modeling dataset. Date and time fields are parsed to engineer response-time features, issue hour, issue weekday, survey weekday, weekend flags, same-day response flags, and response delay indicators. Missing values are handled systematically using median imputation for numeric features and mode imputation for categorical features. Identifier and leakage-prone fields such as unique IDs, order IDs, raw timestamps, customer remarks, and target-derived features are removed from the final modeling matrix. High-cardinality operational fields are handled carefully through aggregate exposure features rather than direct one-hot encoding.

The modeling benchmark first evaluates a 3-class CSAT target: Low, Neutral, and High. This preserves analytical richness and helps study satisfaction gradients. Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting are compared. XGBoost and LightGBM are included defensively if available in the runtime. Because the target is imbalanced, accuracy is not the primary metric. The notebook emphasizes Macro F1, Balanced Accuracy, Macro Recall, Weighted F1, classification reports, and confusion matrices.

The validated benchmark identifies Random Forest as the strongest model among the evaluated classifiers. Hyperparameter tuning is performed using RandomizedSearchCV with stratified cross-validation. Explainability outputs include native feature importance, permutation importance, and class-specific drivers using a one-vs-rest logistic model. SHAP analysis is attempted only when the required package is available.

The final business recommendation comes from target formulation analysis. While the 3-class target is valuable for analysis, the Neutral class is highly imbalanced and difficult to learn reliably. The notebook compares Low / Neutral / High, High vs Non-High, and Low vs Not-Low. The recommended production formulation is **Low vs Not-Low**, because it directly supports Flipkart's operational goal: identify dissatisfied customers who may require escalation, service recovery, or supervisor review. The final deployment recommendation is a Random Forest-based, human-in-the-loop decision support system focused on reducing low-CSAT cases.


## Problem Statement

Flipkart customer support interactions contain signals about customer satisfaction, but dissatisfaction is often discovered only after a low CSAT score is submitted. The business challenge is to use available support interaction data to predict satisfaction outcomes and identify customers likely to become dissatisfied early enough for intervention.

This notebook addresses two linked objectives. First, it benchmarks a 3-class satisfaction model that classifies interactions as Low, Neutral, or High CSAT. Second, it evaluates whether a binary production formulation can provide stronger operational value. The final goal is to recommend a model and target formulation that support proactive, human-in-the-loop customer recovery workflows.


## Business Objective

Build a reliable predictive workflow that helps Flipkart customer experience teams prioritize support interactions at risk of low satisfaction. The model should support escalation, coaching, quality review, and service recovery. It should not be used as an automated punitive system for agents or as a fully automated customer decision engine.


# 1. Import Libraries


In [ ]:
import importlib.util
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42


# 2. Dataset Loading and Initial Audit


In [ ]:
file_path = "Customer_support_data.csv"
df = pd.read_csv(file_path)
raw_df = df.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


In [ ]:
data_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=True).values,
})

print(f"Duplicate rows: {df.duplicated().sum():,}")
data_audit.sort_values("missing_percent", ascending=False)


## Data Quality Interpretation

The target field `CSAT Score` is complete, which makes supervised modeling feasible. Several explanatory fields have high missingness, especially `connected_handling_time`, `Customer_City`, `Product_category`, `Item_price`, and `order_date_time`. These fields must be handled carefully to avoid unstable modeling assumptions. Identifier columns and target-derived fields must also be excluded to prevent leakage.


# 3. Feature Engineering and Data Pre-processing


## Target Selection for Analytical Benchmark

The first modeling benchmark uses a 3-class target:

- `Low`: CSAT scores 1 and 2
- `Neutral`: CSAT score 3
- `High`: CSAT scores 4 and 5

This benchmark preserves analytical richness and allows the notebook to study low, neutral, and high satisfaction outcomes separately. Because the Neutral class is highly imbalanced, a later section compares alternative binary formulations and recommends the final production target.


In [ ]:
df_clean = df.copy()

df_clean["Issue_reported at"] = pd.to_datetime(df_clean["Issue_reported at"], errors="coerce", dayfirst=True)
df_clean["issue_responded"] = pd.to_datetime(df_clean["issue_responded"], errors="coerce", dayfirst=True)
df_clean["Survey_response_Date"] = pd.to_datetime(df_clean["Survey_response_Date"], errors="coerce", dayfirst=True)

df_clean["response_time_minutes"] = (
    df_clean["issue_responded"] - df_clean["Issue_reported at"]
).dt.total_seconds() / 60

invalid_response_count = (df_clean["response_time_minutes"] < 0).sum()
df_clean.loc[df_clean["response_time_minutes"] < 0, "response_time_minutes"] = np.nan

# Retain the validated non-negative response-time feature used in the completed workflow.
df_clean["valid_response_time_minutes"] = df_clean["response_time_minutes"].where(df_clean["response_time_minutes"] >= 0)

invalid_price_count = 0
if "Item_price" in df_clean.columns:
    invalid_price_count = (df_clean["Item_price"] < 0).sum()
    df_clean.loc[df_clean["Item_price"] < 0, "Item_price"] = np.nan

df_clean["csat_target"] = pd.cut(
    df_clean["CSAT Score"],
    bins=[0, 2, 3, 5],
    labels=["Low", "Neutral", "High"],
    include_lowest=True,
)

print(f"Invalid negative response times corrected to missing: {invalid_response_count:,}")
print(f"Invalid negative item prices corrected to missing: {invalid_price_count:,}")
df_clean["csat_target"].value_counts().to_frame("count")


In [ ]:
df_clean["issue_hour"] = df_clean["Issue_reported at"].dt.hour
df_clean["issue_weekday"] = df_clean["Issue_reported at"].dt.day_name()
df_clean["survey_weekday"] = df_clean["Survey_response_Date"].dt.day_name()
df_clean["survey_day"] = df_clean["Survey_response_Date"].dt.strftime("%Y-%m-%d")
df_clean["is_weekend_issue"] = df_clean["Issue_reported at"].dt.dayofweek.isin([5, 6]).astype("int")
df_clean["same_day_response"] = (
    df_clean["Issue_reported at"].dt.date == df_clean["issue_responded"].dt.date
).astype("int")

df_clean["response_delay_flag"] = np.select(
    [
        df_clean["response_time_minutes"].isna(),
        df_clean["response_time_minutes"] <= 15,
        df_clean["response_time_minutes"] <= 60,
        df_clean["response_time_minutes"] <= 240,
        df_clean["response_time_minutes"] > 240,
    ],
    ["Unknown", "Fast", "Moderate", "Delayed", "Severely Delayed"],
    default="Unknown",
)

df_clean["response_time_bucket"] = pd.cut(
    df_clean["response_time_minutes"],
    bins=[0, 5, 15, 60, 240, np.inf],
    labels=["0-5 min", "5-15 min", "15-60 min", "1-4 hrs", ">4 hrs"],
    include_lowest=True,
).astype("object")
df_clean["response_time_bucket"] = df_clean["response_time_bucket"].fillna("Unknown")

df_clean["has_customer_remark"] = df_clean["Customer Remarks"].notna().astype("int")
df_clean["has_order_id"] = df_clean["Order_id"].notna().astype("int")
df_clean["agent_interaction_count"] = df_clean.groupby("Agent_name")["Agent_name"].transform("count")
df_clean["agent_historical_avg_csat"] = df_clean.groupby("Agent_name")["CSAT Score"].transform("mean")
df_clean["supervisor_interaction_count"] = df_clean.groupby("Supervisor")["Supervisor"].transform("count")
df_clean["manager_interaction_count"] = df_clean.groupby("Manager")["Manager"].transform("count")
df_clean["city_interaction_count"] = df_clean.groupby("Customer_City")["Customer_City"].transform("count").fillna(0)

df_clean[["response_time_minutes", "response_time_bucket", "issue_hour", "issue_weekday", "response_delay_flag"]].head()


## Leakage, Missing Value, and Encoding Decisions

Identifiers, raw timestamps, text fields, sparse fields, and high-cardinality raw identifiers are removed or represented through safer engineered features. `agent_historical_avg_csat` is explicitly removed from the modeling matrix because it is target-derived and could leak label information if computed on the full dataset. Numeric missing values are imputed with medians, while categorical missing values are imputed with the most frequent category.


In [ ]:
leakage_or_identifier_cols = [
    "Unique id", "Order_id", "Customer Remarks", "Issue_reported at",
    "issue_responded", "Survey_response_Date", "CSAT Score",
]
sparse_cols_to_drop = ["connected_handling_time", "order_date_time"]
high_cardinality_cols_to_drop = ["Customer_City", "Agent_name", "Supervisor", "Manager"]
columns_to_remove = [
    col for col in leakage_or_identifier_cols + sparse_cols_to_drop + high_cardinality_cols_to_drop
    if col in df_clean.columns
]

y = df_clean["csat_target"].astype("category")
X_raw = df_clean.drop(columns=columns_to_remove + ["csat_target"], errors="ignore")
X_raw = X_raw.drop(columns=["agent_historical_avg_csat"], errors="ignore")

removed_features_summary = pd.DataFrame({
    "removed_column": columns_to_remove + ["agent_historical_avg_csat"],
    "reason": ["identifier/leakage/sparse/high-cardinality raw field"] * len(columns_to_remove)
    + ["target-derived aggregate feature"],
})

numeric_features = X_raw.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_raw.select_dtypes(exclude=["number", "bool"]).columns.tolist()

X_numeric = X_raw[numeric_features].copy()
for col in numeric_features:
    X_numeric[col] = X_numeric[col].fillna(X_numeric[col].median())

X_categorical = X_raw[categorical_features].copy()
for col in categorical_features:
    mode_value = X_categorical[col].mode(dropna=True)
    fill_value = mode_value.iloc[0] if not mode_value.empty else "Unknown"
    X_categorical[col] = X_categorical[col].fillna(fill_value).astype(str)

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

if categorical_features:
    encoded_array = encoder.fit_transform(X_categorical)
    encoded_feature_names = encoder.get_feature_names_out(categorical_features)
    X_encoded_categorical = pd.DataFrame(encoded_array, columns=encoded_feature_names, index=X_categorical.index)
else:
    X_encoded_categorical = pd.DataFrame(index=X_raw.index)

X = pd.concat([X_numeric, X_encoded_categorical], axis=1)
y = y.astype(str)

df_clean = X.copy()
df_clean["csat_target"] = y.values

print(f"Final encoded X shape: {X.shape}")
print(f"Final df_clean shape: {df_clean.shape}")
print(f"Missing values in X: {X.isna().sum().sum():,}")
removed_features_summary


# 4. Class Imbalance and Train-Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

class_distribution = pd.DataFrame({
    "count": y.value_counts(),
    "percentage": (y.value_counts(normalize=True) * 100).round(2),
}).rename_axis("csat_target").reset_index()

imbalance_ratio = class_distribution["count"].max() / class_distribution["count"].min()
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
class_distribution


## Evaluation Strategy

Accuracy alone is insufficient because the High class dominates the target distribution. A model can appear accurate while failing to detect Low or Neutral cases. The comparison therefore uses Macro F1 as the primary metric, Weighted F1 as a secondary metric, and also reports Balanced Accuracy, Macro Precision, Macro Recall, classification reports, and confusion matrices.


# 5. ML Model Implementation and Comparison


In [ ]:
optional_model_status = []
XGBOOST_AVAILABLE = False
LIGHTGBM_AVAILABLE = False

if importlib.util.find_spec("xgboost") is not None:
    try:
        from xgboost import XGBClassifier
        XGBOOST_AVAILABLE = True
        optional_model_status.append({"model": "XGBoost", "status": "Available"})
    except Exception as exc:
        optional_model_status.append({"model": "XGBoost", "status": f"Skipped - {type(exc).__name__}: {exc}"})
else:
    optional_model_status.append({"model": "XGBoost", "status": "Skipped - package unavailable"})

if importlib.util.find_spec("lightgbm") is not None:
    try:
        from lightgbm import LGBMClassifier
        LIGHTGBM_AVAILABLE = True
        optional_model_status.append({"model": "LightGBM", "status": "Available"})
    except Exception as exc:
        optional_model_status.append({"model": "LightGBM", "status": f"Skipped - {type(exc).__name__}: {exc}"})
else:
    optional_model_status.append({"model": "LightGBM", "status": "Skipped - package unavailable"})

pd.DataFrame(optional_model_status)


In [ ]:
models = {
    "Logistic Regression": {
        "estimator": make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False,
    },
    "Decision Tree": {
        "estimator": DecisionTreeClassifier(
            class_weight="balanced", max_depth=12, min_samples_leaf=30, random_state=RANDOM_STATE
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False,
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(
            n_estimators=80, class_weight="balanced_subsample", max_depth=18,
            min_samples_leaf=10, n_jobs=1, random_state=RANDOM_STATE
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False,
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingClassifier(
            n_estimators=60, learning_rate=0.08, max_depth=3, random_state=RANDOM_STATE
        ),
        "uses_sample_weight": True,
        "requires_encoded_target": False,
    },
}

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

if XGBOOST_AVAILABLE:
    models["XGBoost"] = {
        "estimator": XGBClassifier(
            n_estimators=80, max_depth=4, learning_rate=0.08, subsample=0.9,
            colsample_bytree=0.9, objective="multi:softprob", eval_metric="mlogloss",
            random_state=RANDOM_STATE, n_jobs=1
        ),
        "uses_sample_weight": True,
        "requires_encoded_target": True,
    }

if LIGHTGBM_AVAILABLE:
    models["LightGBM"] = {
        "estimator": LGBMClassifier(
            n_estimators=100, learning_rate=0.06, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=1, verbose=-1
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False,
    }

list(models.keys())


In [ ]:
def evaluate_predictions(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

model_results = []
trained_models = {}
classification_reports = {}
confusion_matrices = {}
class_labels = sorted(y.unique())
sample_weights_train = compute_sample_weight(class_weight="balanced", y=y_train)
sample_weights_train_encoded = compute_sample_weight(class_weight="balanced", y=y_train_encoded)

for model_name, config in models.items():
    estimator = config["estimator"]
    start_time = time.time()
    if config["requires_encoded_target"]:
        if config["uses_sample_weight"]:
            estimator.fit(X_train, y_train_encoded, sample_weight=sample_weights_train_encoded)
        else:
            estimator.fit(X_train, y_train_encoded)
        y_pred = label_encoder.inverse_transform(estimator.predict(X_test).astype(int))
    else:
        if config["uses_sample_weight"]:
            estimator.fit(X_train, y_train, sample_weight=sample_weights_train)
        else:
            estimator.fit(X_train, y_train)
        y_pred = estimator.predict(X_test)

    metrics = evaluate_predictions(model_name, y_test, y_pred)
    metrics["Training Time Seconds"] = round(time.time() - start_time, 2)
    model_results.append(metrics)
    trained_models[model_name] = estimator
    classification_reports[model_name] = classification_report(
        y_test, y_pred, labels=class_labels, zero_division=0, output_dict=True
    )
    confusion_matrices[model_name] = confusion_matrix(y_test, y_pred, labels=class_labels)

comparison_table = pd.DataFrame(model_results).sort_values(
    ["Macro F1", "Weighted F1"], ascending=False
).reset_index(drop=True)
comparison_table


## Model Comparison Interpretation

The comparison table ranks models by Macro F1 first and Weighted F1 second. This prevents the majority High class from dominating model selection. In the validated workflow, Random Forest is the strongest baseline model because it provides the best balance of minority-class detection and overall predictive performance.


In [ ]:
for model_name in comparison_table["Model"]:
    print("=" * 90)
    print(f"Classification Report: {model_name}")
    print("=" * 90)
    display(pd.DataFrame(classification_reports[model_name]).T.round(3))


In [ ]:
plt.close("all")
num_models = len(comparison_table)
cols = 2
rows = int(np.ceil(num_models / cols))
fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, model_name in zip(axes, comparison_table["Model"]):
    disp = ConfusionMatrixDisplay(confusion_matrices[model_name], display_labels=class_labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Confusion Matrix: {model_name}")

for ax in axes[num_models:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


# 6. Cross-Validation and Hyperparameter Tuning


In [ ]:
best_model_name = comparison_table.loc[0, "Model"]
best_model = trained_models[best_model_name]
print(f"Best baseline model: {best_model_name}")
comparison_table.head(1)


In [ ]:
def get_tuning_setup(model_name):
    if model_name == "Logistic Regression":
        estimator = make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1200, random_state=RANDOM_STATE),
        )
        params = {
            "logisticregression__C": uniform(0.05, 5.0),
            "logisticregression__solver": ["lbfgs", "liblinear"],
        }
        return estimator, params, False, False
    if model_name == "Decision Tree":
        estimator = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)
        params = {
            "max_depth": randint(4, 25),
            "min_samples_leaf": randint(10, 120),
            "min_samples_split": randint(20, 200),
            "criterion": ["gini", "entropy"],
        }
        return estimator, params, False, False
    if model_name == "Random Forest":
        estimator = RandomForestClassifier(class_weight="balanced_subsample", n_jobs=1, random_state=RANDOM_STATE)
        params = {
            "n_estimators": randint(60, 140),
            "max_depth": randint(8, 26),
            "min_samples_leaf": randint(5, 60),
            "max_features": ["sqrt", "log2", None],
        }
        return estimator, params, False, False
    if model_name == "Gradient Boosting":
        estimator = GradientBoostingClassifier(random_state=RANDOM_STATE)
        params = {
            "n_estimators": randint(40, 100),
            "learning_rate": uniform(0.03, 0.12),
            "max_depth": randint(2, 5),
            "min_samples_leaf": randint(10, 80),
        }
        return estimator, params, True, False
    raise ValueError(f"No tuning setup available for {model_name}")

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
tuning_estimator, param_distributions, uses_sample_weight, requires_encoded_target = get_tuning_setup(best_model_name)

search = RandomizedSearchCV(
    estimator=tuning_estimator,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="f1_macro",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=0,
)

if requires_encoded_target:
    fit_y = y_train_encoded
    if uses_sample_weight:
        search.fit(X_train, fit_y, sample_weight=sample_weights_train_encoded)
    else:
        search.fit(X_train, fit_y)
else:
    if uses_sample_weight:
        search.fit(X_train, y_train, sample_weight=sample_weights_train)
    else:
        search.fit(X_train, y_train)

tuned_best_model = search.best_estimator_
print(f"Tuned model: {best_model_name}")
print(f"Best CV Macro F1: {search.best_score_:.4f}")
print("Best hyperparameters:")
print(search.best_params_)


In [ ]:
if requires_encoded_target:
    tuned_y_pred = label_encoder.inverse_transform(tuned_best_model.predict(X_test).astype(int))
else:
    tuned_y_pred = tuned_best_model.predict(X_test)

final_test_metrics = evaluate_predictions(f"Tuned {best_model_name}", y_test, tuned_y_pred)
final_test_metrics["Best CV Macro F1"] = search.best_score_

print(classification_report(y_test, tuned_y_pred, labels=class_labels, zero_division=0))
final_test_performance = pd.DataFrame([final_test_metrics])
final_test_performance


In [ ]:
plt.close("all")
tuned_cm = confusion_matrix(y_test, tuned_y_pred, labels=class_labels)
ConfusionMatrixDisplay(tuned_cm, display_labels=class_labels).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix: Tuned {best_model_name}")
plt.tight_layout()
plt.show()


# 7. Feature Importance and Explainability


In [ ]:
def get_feature_importance(estimator, feature_names):
    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        importance_type = "native_feature_importance"
    elif hasattr(estimator, "named_steps") and "logisticregression" in estimator.named_steps:
        values = np.abs(estimator.named_steps["logisticregression"].coef_).mean(axis=0)
        importance_type = "mean_absolute_logistic_coefficient"
    elif hasattr(estimator, "coef_"):
        values = np.abs(estimator.coef_).mean(axis=0)
        importance_type = "mean_absolute_coefficient"
    else:
        return pd.DataFrame(columns=["feature", "importance", "importance_type"])
    return pd.DataFrame({
        "feature": feature_names,
        "importance": values,
        "importance_type": importance_type,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

feature_importance_df = get_feature_importance(tuned_best_model, X.columns)
feature_importance_df.head(20)


In [ ]:
plt.close("all")
if not feature_importance_df.empty:
    plot_df = feature_importance_df.head(20).sort_values("importance", ascending=True)
    ax = sns.barplot(data=plot_df, x="importance", y="feature", palette="viridis")
    ax.set_title(f"Top 20 Feature Importances - Tuned {best_model_name}")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("Native feature importance is not available for the selected model.")


In [ ]:
permutation_sample_size = min(5000, X_test.shape[0])
X_perm = X_test.sample(n=permutation_sample_size, random_state=RANDOM_STATE)
y_perm = y_test.loc[X_perm.index]

permutation_result = permutation_importance(
    tuned_best_model,
    X_perm,
    y_perm,
    scoring="f1_macro",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

permutation_importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": permutation_result.importances_mean,
    "importance_std": permutation_result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

permutation_importance_df.head(20)


In [ ]:
plt.close("all")
plot_df = permutation_importance_df.head(20).sort_values("importance_mean", ascending=True)
ax = sns.barplot(data=plot_df, x="importance_mean", y="feature", palette="mako")
ax.set_title(f"Top 20 Permutation Importances - Tuned {best_model_name}")
ax.set_xlabel("Mean Macro F1 Decrease")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
shap_status = "Not attempted"
shap_top_features = pd.DataFrame()

if importlib.util.find_spec("shap") is not None:
    try:
        import shap
        shap_sample_size = min(1000, X_test.shape[0])
        X_shap = X_test.sample(n=shap_sample_size, random_state=RANDOM_STATE)
        if hasattr(tuned_best_model, "feature_importances_"):
            explainer = shap.TreeExplainer(tuned_best_model)
            shap_values = explainer.shap_values(X_shap)
            if isinstance(shap_values, list):
                mean_abs_shap = np.mean([np.abs(values).mean(axis=0) for values in shap_values], axis=0)
            else:
                shap_array = np.array(shap_values)
                mean_abs_shap = np.abs(shap_array).mean(axis=(0, 2)) if shap_array.ndim == 3 else np.abs(shap_array).mean(axis=0)
            shap_top_features = pd.DataFrame({
                "feature": X.columns,
                "mean_abs_shap": mean_abs_shap,
            }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
            shap_status = "Completed"
        else:
            shap_status = "Skipped - selected model is not a direct tree estimator"
    except Exception as exc:
        shap_status = f"Skipped - {type(exc).__name__}: {exc}"
else:
    shap_status = "Skipped - shap package unavailable"

print(shap_status)
shap_top_features.head(20)


## Class-Specific Drivers

A one-vs-rest logistic model identifies directional class-specific drivers. Positive coefficients indicate features associated with a higher likelihood of that class relative to the other classes. These are association-based explanations, not causal claims.


In [ ]:
ovr_model = OneVsRestClassifier(
    make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
)
ovr_model.fit(X_train, y_train)

class_driver_tables = {}
for class_label, estimator in zip(ovr_model.classes_, ovr_model.estimators_):
    coefficients = estimator.named_steps["logisticregression"].coef_.ravel()
    class_driver_tables[class_label] = pd.DataFrame({
        "feature": X.columns,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients),
    }).sort_values("coefficient", ascending=False).reset_index(drop=True)

for class_label in class_labels:
    print("=" * 90)
    print(f"Top positive drivers for {class_label} CSAT")
    print("=" * 90)
    display(class_driver_tables[class_label].head(10)[["feature", "coefficient"]].round(4))


# 8. Target Formulation Evaluation


The 3-class target is analytically useful, but the Neutral class is highly imbalanced. For production, the model should support a clear business action. This section compares three formulations: Low / Neutral / High, High vs Non-High, and Low vs Not-Low.


In [ ]:
def build_formulation_target(csat_scores, formulation):
    if formulation == "3-Class Low/Neutral/High":
        return pd.cut(csat_scores, bins=[0, 2, 3, 5], labels=["Low", "Neutral", "High"], include_lowest=True).astype(str)
    if formulation == "High vs Non-High":
        return pd.Series(np.where(csat_scores >= 4, "High", "Non-High"), index=csat_scores.index)
    if formulation == "Low vs Not-Low":
        return pd.Series(np.where(csat_scores <= 2, "Low", "Not-Low"), index=csat_scores.index)
    raise ValueError(formulation)

def minority_recall_score(y_true, y_pred):
    counts = pd.Series(y_true).value_counts()
    minority_label = counts.idxmin()
    labels = sorted(pd.Series(y_true).unique())
    recalls = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    return minority_label, dict(zip(labels, recalls))[minority_label]

def evaluate_target_formulation(formulation_name, target_series):
    X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
        X, target_series.astype(str), test_size=0.20, random_state=RANDOM_STATE, stratify=target_series.astype(str)
    )
    candidate_models = {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        "Random Forest": RandomForestClassifier(n_estimators=80, class_weight="balanced_subsample", max_depth=18, min_samples_leaf=10, n_jobs=1, random_state=RANDOM_STATE),
    }
    rows = []
    for model_name, estimator in candidate_models.items():
        estimator.fit(X_train_f, y_train_f)
        y_pred_f = estimator.predict(X_test_f)
        minority_label, minority_recall = minority_recall_score(y_test_f, y_pred_f)
        row = evaluate_predictions(f"{formulation_name} - {model_name}", y_test_f, y_pred_f)
        row.update({
            "Formulation": formulation_name,
            "Base Model": model_name,
            "Class Count": target_series.nunique(),
            "Minority Class": minority_label,
            "Minority Recall": minority_recall,
            "Majority Class Share": target_series.value_counts(normalize=True).max(),
        })
        rows.append(row)
    return rows

formulation_names = ["3-Class Low/Neutral/High", "High vs Non-High", "Low vs Not-Low"]
formulation_results = []
for formulation_name in formulation_names:
    target_series = build_formulation_target(raw_df["CSAT Score"], formulation_name)
    formulation_results.extend(evaluate_target_formulation(formulation_name, target_series))

formulation_comparison_table = pd.DataFrame(formulation_results).sort_values(
    ["Macro F1", "Weighted F1"], ascending=False
).reset_index(drop=True)
selected_cols = [
    "Formulation", "Base Model", "Class Count", "Accuracy", "Balanced Accuracy",
    "Macro F1", "Weighted F1", "Minority Class", "Minority Recall", "Majority Class Share",
]
formulation_comparison_table[selected_cols]


In [ ]:
best_by_formulation = (
    formulation_comparison_table
    .sort_values(["Formulation", "Macro F1", "Weighted F1"], ascending=[True, False, False])
    .groupby("Formulation", as_index=False)
    .head(1)
    .sort_values(["Macro F1", "Weighted F1"], ascending=False)
    .reset_index(drop=True)
)

recommended_formulation = "Low vs Not-Low"
recommended_row = best_by_formulation[best_by_formulation["Formulation"] == recommended_formulation].iloc[0]
best_by_formulation[selected_cols]


## Target Formulation Recommendation

The recommended production formulation is **Low vs Not-Low**.

The 3-class formulation preserves analytical richness and is useful for diagnostics, but the Neutral class is highly imbalanced and difficult to predict reliably. High vs Non-High can perform well numerically, but it blends neutral customers with dissatisfied customers. Low vs Not-Low directly identifies dissatisfied customers requiring intervention, making it the most useful operational target for Flipkart leadership.


# 9. Executive Summary and Final Recommendation


In [ ]:
executive_summary = pd.DataFrame({
    "Section": [
        "Objective",
        "Dataset size",
        "Analytical benchmark target",
        "Production target recommendation",
        "Best benchmark model",
        "Tuned benchmark Macro F1",
        "Tuned benchmark Weighted F1",
        "Recommended formulation Macro F1",
        "Recommended formulation Weighted F1",
        "Deployment recommendation",
    ],
    "Summary": [
        "Predict customer satisfaction outcomes and identify dissatisfied customers proactively.",
        f"{df.shape[0]:,} rows and {df.shape[1]:,} raw columns.",
        "3-Class Low / Neutral / High",
        recommended_formulation,
        best_model_name,
        f"{final_test_metrics['Macro F1']:.4f}",
        f"{final_test_metrics['Weighted F1']:.4f}",
        f"{recommended_row['Macro F1']:.4f}",
        f"{recommended_row['Weighted F1']:.4f}",
        "Random Forest Low vs Not-Low as a human-in-the-loop decision support system.",
    ],
})
executive_summary


## Business Recommendations

- Use the Low vs Not-Low model to flag interactions at risk of low CSAT.
- Prioritize flagged interactions for supervisor review, escalation, or service recovery.
- Monitor delayed and severely delayed response patterns as operational risk signals.
- Strengthen agent onboarding and coaching for segments associated with low CSAT.
- Improve channel-specific playbooks, especially where email or other lower-performing channels appear as risk drivers.
- Improve source-system capture for sparse fields such as handling time, product category, item price, and city.
- Treat predictions as decision support, not as automated punitive judgments.


# 10. Production Deployment Considerations

## Monitoring

Monitor Macro F1, Balanced Accuracy, Weighted F1, Low-class recall, and false negatives among dissatisfied customers. Track prediction rates by channel, issue category, tenure bucket, shift, and major city groups.

## Retraining Frequency

Start with monthly retraining when new CSAT labels are available. Retrain more frequently during large sale periods, policy changes, support workflow changes, or detected drift.

## Data Drift Detection

Track drift in support channel mix, issue categories, response-time distribution, product categories, city coverage, missing-value rates, and predicted Low probability. Use PSI, distribution checks, or dashboard thresholds depending on platform maturity.

## Fairness Considerations

The model should not be used to unfairly penalize agents, shifts, cities, or teams. Some groups may handle more complex cases. Monitor error rates and prediction rates across operational segments and keep a human review layer in the workflow.

## Limitations

The current dataset covers a limited period, several fields have high missingness, customer remarks are not modeled with NLP, and the validation split is random rather than time-based. Feature importance identifies associations, not causal proof.

## Deployment Recommendation

Deploy the Random Forest Low vs Not-Low model as a human-in-the-loop decision support system. The model should trigger review and recovery workflows, not fully automated decisions.


# 11. Future Work

Recommended future improvements include:

1. NLP using customer remarks.
2. Real-time prediction pipelines.
3. Ensemble approaches.
4. Human-in-the-loop escalation workflows.
5. Probability calibration and threshold optimization.
6. Time-based validation using future months of data.
7. Monitoring dashboards for drift, fairness, and low-CSAT recall.


# 12. Conclusion

This notebook builds a complete machine learning workflow for Flipkart customer satisfaction prediction. It prepares an ML-ready dataset, benchmarks multiple classification models, handles class imbalance, performs hyperparameter tuning, generates explainability outputs, compares target formulations, and recommends a deployment strategy.

Random Forest is retained as the best-performing model from the validated benchmark. The 3-class Low / Neutral / High formulation is valuable for analytical understanding, but Low vs Not-Low is recommended for deployment because it directly identifies dissatisfied customers requiring intervention. The final solution is best positioned as a responsible, human-in-the-loop decision support system for improving customer satisfaction and reducing low-CSAT cases.
